# Model Visualization Notebook

This notebook creates three presentation-ready visuals for the sentiment classification project:

1. Bar chart for model accuracy and F1-score
2. Confusion matrix heatmaps
3. Word clouds for positive and negative reviews


In [ ]:
from pathlib import Path
from collections import Counter
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.tree import DecisionTreeClassifier
from wordcloud import WordCloud

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 150

ROOT = Path.cwd()
CANDIDATES = [
    ROOT / 'train_fixed_split.csv',
    ROOT.parent / 'train_fixed_split.csv',
    ROOT.parent / '527_project' / 'train_fixed_split.csv',
]

def locate_file(name: str) -> Path:
    for candidate in [ROOT / name, ROOT.parent / name, ROOT.parent / '527_project' / name]:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Cannot find {name}')

train_path = locate_file('train_fixed_split.csv')
test_path = locate_file('test_fixed_split.csv')

train_df = pd.read_csv(train_path).dropna(subset=['text']).copy()
test_df = pd.read_csv(test_path).dropna(subset=['text']).copy()

train_df.head()


In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, min_df=2, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_df['text'])
X_test = vectorizer.transform(test_df['text'])
y_train = train_df['label']
y_test = test_df['label']

models = {
    'Logistic Regression (Baseline)': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, random_state=42),
}

results = []
conf_mats = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'F1': f1_score(y_test, pred, zero_division=0),
    })
    conf_mats[name] = confusion_matrix(y_test, pred)

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
results_df


In [ ]:
palette = {
    'Accuracy': '#E3A56D',
    'F1': '#F4D03F',
}

plot_df = results_df.melt(id_vars='Model', value_vars=['Accuracy', 'F1'], var_name='Metric', value_name='Score')

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.barplot(data=plot_df, x='Model', y='Score', hue='Metric', palette=palette, ax=ax)
ax.set_title('Model Accuracy and F1-score Comparison', fontsize=16, weight='bold')
ax.set_xlabel('')
ax.set_ylabel('Score')
ax.set_ylim(0.86, 0.97)
ax.tick_params(axis='x', rotation=15)

for container in ax.containers:
    ax.bar_label(container, fmt='%.4f', padding=3, fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for ax, (name, matrix) in zip(axes, conf_mats.items()):
    sns.heatmap(
        matrix,
        annot=True,
        fmt='d',
        cmap=sns.light_palette('#C45C3E', as_cmap=True),
        cbar=False,
        xticklabels=['Pred Neg', 'Pred Pos'],
        yticklabels=['True Neg', 'True Pos'],
        ax=ax,
    )
    ax.set_title(name, fontsize=12, weight='bold')

plt.suptitle('Confusion Matrix Comparison', fontsize=16, weight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
custom_stopwords = set(ENGLISH_STOP_WORDS) | {
    'album', 'albums', 'dark', 'floyd', 'good', 'great', 'just', 'like', 'moon',
    'movie', 'movies', 'music', 'netflix', 'one', 'pink', 'que', 'really', 'series',
    'show', 'shows', 'song', 'songs', 'story', 'time', 'watch', 'watched', 'watching'
}

def prepare_text(series):
    tokens = []
    for text in series.astype(str):
        tokens.extend([w for w in re.findall(r'[a-zA-Z]{3,}', text.lower()) if w not in custom_stopwords])
    return ' '.join(tokens)

positive_text = prepare_text(train_df.loc[train_df['label'] == 1, 'text'])
negative_text = prepare_text(train_df.loc[train_df['label'] == 0, 'text'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
configs = [
    ('Positive Reviews', positive_text, '#E3A56D'),
    ('Negative Reviews', negative_text, '#C45C3E'),
]

for ax, (title, text, color) in zip(axes, configs):
    wc = WordCloud(width=900, height=500, background_color='#0B0C10', colormap='autumn', collocations=False).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontsize=15, weight='bold', color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()
